# L02 · 场景、实体与仿真生命周期

> **课程状态：** 本实验已在 CPU 和参考 AMD ROCm 平台上通过干净 kernel 验证，其中包括离屏渲染；L02 状态为 `cpu-verified`。

本 notebook 使用一个 Plane 和一个下落的 Box，把讲义中的“声明 → 构建 → 运行”模型转化为可观察证据。请从干净 kernel 开始，按顺序运行全部 cell。


## 运行前须知

仿真后端与渲染开关相互独立：

- `ROBO_GENESIS_BACKEND=auto` 是默认模式；已验证的 AMD ROCm 后端可用时选择它，否则使用 CPU。
- `ROBO_GENESIS_BACKEND=cpu` 强制使用最低要求的 CPU 路径。
- `ROBO_GENESIS_RENDER=0` 关闭相机渲染，改为生成基于状态数据的示意图。
- `ROBO_GENESIS_RENDER=1` 启用离屏相机；此时渲染错误会让运行失败，不会静默降级。

由于 `gs.init()` 是进程级初始化，修改任一设置后都必须使用新的 kernel。


In [ ]:
import os

import genesis as gs
import matplotlib.pyplot as plt
import numpy as np
import torch

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import (
    environment_report,
    notebook_mode,
    select_backend,
    to_numpy,
)

lesson = load_course_manifest().lesson("L02")
print(f"lesson_status: {lesson.status.value}")

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")

render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l02-one-box-lifecycle", show_viewer=False)
output_dir = runtime["output_dir"]
environment = environment_report()
selected_backend = select_backend(prefer_rocm=backend_mode == "auto")

if backend_mode == "cpu":
    selection_reason = "forced CPU mode requested by ROBO_GENESIS_BACKEND"
elif getattr(gs, "amdgpu", None) is not None and selected_backend == gs.amdgpu:
    selection_reason = "auto mode found a verified ROCm device"
else:
    selection_reason = "auto mode found no verified ROCm path; using CPU"

print(f"backend_mode: {backend_mode}")
print(f"selected_backend: {selected_backend}")
print(f"selection_reason: {selection_reason}")
print(f"render_enabled: {render_enabled}")
print(f"output_dir: {output_dir.resolve()}")

gs.init(backend=selected_backend, logging_level="warning")
assert gs.backend == selected_backend
actual_backend = f"{gs.backend}"
print(f"actual_backend: {actual_backend}")


## 1. 声明场景

Plane 和 Box 都是内置 primitive。Box 的声明显式展示 Morph、Material 和 Surface；只有在最终确定拓扑前启用了渲染，才会添加相机。


In [ ]:
BOX_SIZE = (0.10, 0.10, 0.10)
BOX_INITIAL_POS = (0.0, 0.0, 0.50)
BOX_DENSITY = 500.0
BOX_COLOR = (0.20, 0.60, 0.90, 1.0)
CAMERA_RESOLUTION = (640, 360)

scene = gs.Scene(
    show_viewer=runtime["show_viewer"],
    sim_options=gs.options.SimOptions(dt=0.01, substeps=2),
)
ground = scene.add_entity(
    morph=gs.morphs.Plane(),
    name="ground",
)
box = scene.add_entity(
    morph=gs.morphs.Box(size=BOX_SIZE, pos=BOX_INITIAL_POS),
    material=gs.materials.Rigid(rho=BOX_DENSITY),
    surface=gs.surfaces.Default(color=BOX_COLOR),
    name="falling_box",
)

camera = None
if render_enabled:
    camera = scene.add_camera(
        res=CAMERA_RESOLUTION,
        pos=(1.1, -1.1, 0.8),
        lookat=(0.0, 0.0, 0.25),
        fov=40,
        GUI=False,
    )

assert scene.is_built is False
assert (camera is not None) == render_enabled
print(f"scene.is_built: {scene.is_built}")
print(f"entity handle: {type(box).__name__}")
print(f"Morph: Box(size={BOX_SIZE}, pos={BOX_INITIAL_POS})")
print(f"Material: Rigid(rho={BOX_DENSITY})")
print(f"Surface: Default(color={BOX_COLOR})")
print(f"camera declared: {camera is not None}")


## 2. 检验 build 前边界

Box handle 已经存在，但动态状态必须仍然不可用。只有捕获到预期的 Genesis 异常，并且消息内容正确，才算通过。


In [ ]:
pre_build_guard_passed = False
try:
    box.get_pos()
except gs.GenesisException as exc:
    if "not built yet" not in str(exc).lower():
        raise
    pre_build_guard_passed = True
    print("PASS — state is unavailable before build")
else:
    raise AssertionError("get_pos unexpectedly succeeded before scene.build()")


## 3. 只 build 一次并检查层级

build 调用会跨过生命周期边界。构建成功后，应当检查当前 primitive 的实际结构，不能假设每个刚体实体都有相同数量的 link 或碰撞 geom。


In [ ]:
scene.build()
assert scene.is_built is True
print("PASS — scene.is_built changed from False to True")


In [ ]:
entity_type = type(box).__name__
link_count = box.n_links
link_list_count = len(box.links)
geom_count = box.n_geoms
geom_list_count = len(box.geoms)

assert entity_type == "RigidEntity"
assert link_count == link_list_count == 1
assert geom_count == geom_list_count == 1
print(
    "PASS — this Box primitive has "
    f"{link_count} RigidLink and {geom_count} collision RigidGeom"
)
print("This count describes this primitive, not every Genesis Entity.")


## 4. 步进并检查状态

在恰好 20 个外层 step 前后分别记录位置、四元数和线速度。检查依据是 shape、device、数值有限性和方向性预测，而不是编造一个精确的最终坐标。


In [ ]:
def checked_state(name, value, expected_shape):
    if not isinstance(value, torch.Tensor):
        raise TypeError(f"{name} must be a torch.Tensor, got {type(value).__name__}")
    actual_shape = tuple(value.shape)
    if actual_shape != expected_shape:
        raise AssertionError(f"{name} shape {actual_shape} != {expected_shape}")
    if not bool(torch.isfinite(value).all().item()):
        raise AssertionError(f"{name} contains non-finite values: {value}")
    print(f"PASS — {name}: shape={actual_shape}, device={value.device}, finite=True")
    return value.detach().clone()


initial_pos = checked_state("initial_pos", box.get_pos(), (3,))
initial_quat = checked_state("initial_quat", box.get_quat(), (4,))
initial_vel = checked_state("initial_vel", box.get_vel(), (3,))
trajectory = [initial_pos]

for _ in range(20):
    scene.step()
    trajectory.append(box.get_pos().detach().clone())

final_pos = checked_state("final_pos", box.get_pos(), (3,))
final_quat = checked_state("final_quat", box.get_quat(), (4,))
final_vel = checked_state("final_vel", box.get_vel(), (3,))
initial_z = float(initial_pos[2].detach().cpu())
final_z = float(final_pos[2].detach().cpu())
if not final_z < initial_z:
    raise AssertionError(f"expected final_z < initial_z, got {final_z} >= {initial_z}")
height_drop = initial_z - final_z
print(f"PASS — Box height decreased by {height_drop:.6f} m over 20 steps")


## 5. 不夸大观测证据

启用渲染时，验证并显示返回的 RGB 数据；关闭渲染时，明确报告 skip，并且只绘制基于状态的侧视图。两个分支支持的结论并不相同。


In [ ]:
if render_enabled:
    assert camera is not None
    rgb = to_numpy(camera.render(rgb=True)[0])
    if rgb.ndim != 3:
        raise AssertionError(f"RGB array must be 3D, got shape {rgb.shape}")
    expected_width, expected_height = CAMERA_RESOLUTION
    if rgb.shape[:2] != (expected_height, expected_width):
        raise AssertionError(
            f"RGB height/width {rgb.shape[:2]} != {(expected_height, expected_width)}"
        )
    if rgb.shape[2] not in (3, 4):
        raise AssertionError(f"RGB channel count must be 3 or 4, got {rgb.shape[2]}")
    if not np.isfinite(rgb).all():
        raise AssertionError("RGB array contains non-finite values")

    rgb_min = float(rgb.min())
    rgb_max = float(rgb.max())
    if np.issubdtype(rgb.dtype, np.integer):
        dtype_max = float(np.iinfo(rgb.dtype).max)
        if not 0.0 <= rgb_min <= rgb_max <= dtype_max:
            raise AssertionError(f"RGB range [{rgb_min}, {rgb_max}] is invalid for {rgb.dtype}")
    elif np.issubdtype(rgb.dtype, np.floating):
        if not 0.0 <= rgb_min <= rgb_max <= 1.0 + 1e-6:
            raise AssertionError(f"RGB range [{rgb_min}, {rgb_max}] is invalid for {rgb.dtype}")
    else:
        raise TypeError(f"unsupported RGB dtype: {rgb.dtype}")

    observation_path = output_dir / "l02-camera-rgb.png"
    plt.imsave(observation_path, rgb)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(rgb)
    ax.set_title("Genesis offscreen RGB observation")
    ax.axis("off")
    plt.show()
    plt.close(fig)
    render_result = f"PASS: RGB shape={rgb.shape}, dtype={rgb.dtype}, range=[{rgb_min}, {rgb_max}]"
    print(render_result)
else:
    print("SKIP: rendering disabled for this run")
    positions = np.stack([to_numpy(position) for position in trajectory])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(positions[:, 0], positions[:, 2], marker="o", markersize=3)
    ax.axhline(0.0, color="black", linewidth=1, label="Plane z=0")
    ax.set_xlabel("x position (m)")
    ax.set_ylabel("z position (m)")
    ax.set_title("State-derived side view (not a camera render)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    observation_path = output_dir / "l02-state-schematic.png"
    fig.savefig(observation_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    render_result = "SKIP: rendering disabled; state-derived schematic only"
    print(f"schematic: {observation_path.resolve()}")


## 6. 检验已关闭的拓扑

build 完成后，添加新实体必须产生预期的生命周期错误。和前面一样，无关异常不能算作证据。


In [ ]:
post_build_guard_passed = False
try:
    scene.add_entity(
        morph=gs.morphs.Sphere(radius=0.05),
        name="too_late",
    )
except gs.GenesisException as exc:
    if "already built" not in str(exc).lower():
        raise
    post_build_guard_passed = True
    print("PASS — topology declaration is closed after build")
else:
    raise AssertionError("add_entity unexpectedly succeeded after scene.build()")


## 证据汇总

最终汇总会区分核心仿真结果和可选渲染结果。跳过渲染不能被报告成渲染通过。


In [ ]:
assert pre_build_guard_passed
assert post_build_guard_passed
assert scene.is_built is True
assert link_count == geom_count == 1
assert final_z < initial_z

evidence = {
    "genesis_world": environment["genesis_world"],
    "backend_mode": backend_mode,
    "actual_backend": actual_backend,
    "render_enabled": render_enabled,
    "render_result": render_result,
    "scene_built": scene.is_built,
    "entity_type": entity_type,
    "links": link_count,
    "collision_geoms": geom_count,
    "initial_z": initial_z,
    "final_z": final_z,
    "height_drop": height_drop,
    "observation_artifact": str(observation_path),
}
for key, value in evidence.items():
    print(f"{key:>22}: {value}")
print("PASS — core L02 lifecycle evidence is complete")


## 练习

在声明场景的 cell 中、`scene.build()` 之前添加下面这个固定 marker：

```python
marker = scene.add_entity(
    morph=gs.morphs.Box(
        size=(0.06, 0.06, 0.06),
        pos=(0.18, 0.0, 0.03),
        fixed=True,
    ),
    surface=gs.surfaces.Default(color=(0.20, 0.80, 0.35, 1.0)),
    name="fixed_marker",
)
```

重启 kernel 并从头运行 notebook。检查 marker 的实体和 link 结构；启用渲染时，还要确认它出现在图像中。解释哪些参数属于 Morph，哪些属于 Surface。


## 与 L03 的衔接

本实验固定了物理配置，让生命周期成为唯一被考察的变量。L03 将改变时间步长、substeps、接触和摩擦，并测量这些选择如何影响刚体行为。
